# 02b · Fine-tune multilingual-e5-small on the training pairs (contrastive)

This notebook fine-tunes the embedding model that builds the buckets, `intfloat/multilingual-e5-small`, on this
competition's own matches:
* **Positive pair:** an S1 and one of its true S2/S3 records from `train_ground_truth.tsv`.
* **Hard negative:** a lookalike from the same S1's bucket (a candidate 02 full ranked high that is *not* its match).
* **Loss:** `MultipleNegativesRankingLoss` (InfoNCE). In every batch an S1 must be closer to its true record than
  to its hard negative and to every other record in the batch. Each batch holds one country only, so the in-batch
  negatives are records the S1 really competes with.

**Why.** The e5 rank, score and margins are the matcher's strongest features, and e5 top-30 still misses 5.3% of
India's true pairs (02 full's train report). A model trained on this data's noise (typos, reordered addresses,
transliterated Indian-script names, websites as names) should rank true records higher and lookalikes lower.

**Same text as 02 full.** The training texts are built with 02 full's own functions, copied below:
* `name | address` from 01's cleaned fields;
* Indian-script names and addresses transliterated from the original text, names through the word map;
* the `query: ` prefix and a 128-token limit.

The model is trained on exactly the text it will see in 02 full.

**No leakage into LightGBM.** 03 full trains LightGBM on 200,000 random train S1 per country (seed 42). This
notebook recomputes that exact choice and **never fine-tunes on those S1**. Their e5 scores stay honest, and 03's
cross-validation remains a fair estimate. 01, 02 full and 03 full are unchanged.

---

**Inputs (attach on Kaggle)**

| Input | Needed for |
|---|---|
| 01's `processed/` (`FULL_RUN=true`) | required: texts of every train record |
| competition data (`train_ground_truth.tsv`) | required: the positive pairs |
| 02 full's output (`buckets/train_*.parquet`, `translit_wordmap.json`) | recommended: hard negatives, and the exact word map. Without it, only in-batch negatives are used, and the word map is learned here (same function, same result) |

**Outputs** (`/kaggle/working/e5_finetune/`)

| File | Content |
|---|---|
| `e5-small-er/` | the fine-tuned model (SentenceTransformer format) |
| `eval_recall.csv` | e5 top-k recall, base vs fine-tuned, on S1 the model never saw and on S1 it trained on |
| `train_log.csv`, `finetune_config.json` | loss curve and settings |
| `lgbm_train_s1.txt`, `finetune_s1.txt` | 03's LightGBM S1 (never fine-tuned on) and the S1 whose pairs were used here |

**Using the model** (the full pipeline has to run again, since every e5 score changes):
1. Run this notebook on GPU T4 x2 and save a version. Add its output as a dataset.
2. Run `02_full_e5_buckets.ipynb` with `E5_MODELS=/kaggle/input/<this dataset>/e5_finetune/e5-small-er`.
   The buckets go to `embeddings_full/e5-small-er/`.
3. Run `03_full_lightgbm_submission.ipynb` with `BUCKET_DIR=<that run>/embeddings_full/e5-small-er/buckets`,
   keeping its defaults `TRAIN_S1_PER_COUNTRY=200000` and seed 42. That is what keeps LightGBM on unseen S1.
4. Compare 03's cross-validation (0.9796 with the base model) and 02's train report (India top-30 recall 0.9471,
   ceiling 0.9917).

**Caveats**
* The fine-tune S1 are still in the train buckets. They compete with LightGBM's S1 for candidates (`cand_margin`,
  `rev_rank`), and their own matches will sit closer to them than on test. The "seen vs unseen" rows of the
  evaluation measure how much the model memorized. If the gap is small, this effect is small too.
* France has no labels, so the effect there cannot be measured. As a proxy, run once with
  `FINETUNE_COUNTRIES=India` and `EVAL_COUNTRIES=US`: it shows whether fine-tuning carries over to a country the
  model did not train on.
* Recall is measured at full size. Each held-out S1 is searched against **all** train S2/S3 records of its country,
  because samples inflate recall.

## Settings
Each can also be set as an environment variable of the same name.

| Setting | Default | Meaning |
|---|---|---|
| `BASE_MODEL` | `intfloat/multilingual-e5-small` | model to fine-tune |
| `FINETUNE_COUNTRIES` | `India,US` | countries whose pairs are used for training |
| `MAX_PAIRS_PER_COUNTRY` | `500000` | positive pairs per country (random, from S1 outside LightGBM's set) |
| `HARD_NEGATIVES` | `true` | one bucket lookalike per pair as an extra negative (needs 02 full's train buckets) |
| `HARD_NEG_TOP` | `10` | the hard negative is drawn from the S1's 10 best-scored non-matching candidates |
| `EPOCHS`, `BATCH_SIZE`, `LEARNING_RATE` | `1`, `256`, `2e-5` | training (`BATCH_SIZE` rows per step, over all GPUs); warm-up is 10% of the steps |
| `TRAIN_GPUS` | all GPUs | GPUs used for training, one process each (DDP) |
| `MINI_BATCH` | `64` | above 128 rows per GPU, the loss is computed this many rows at a time (same result, less memory) |
| `FINETUNED_MODEL` | empty | folder of a model this notebook already fine-tuned (e.g. an earlier version's `e5_finetune/e5-small-er`, attached as input): skip training, evaluate it |
| `SCALE` | `20` | inverse temperature of the loss (cosine × 20) |
| `LGBM_SEED`, `LGBM_TRAIN_S1_PER_COUNTRY` | `42`, `200000` | must equal 03 full's `SEED` and `TRAIN_S1_PER_COUNTRY` |
| `EVAL_COUNTRIES` | `India` | countries evaluated at full size (US adds ~6M records to encode per model) |
| `EVAL_S1`, `SEEN_EVAL_S1` | `20000`, `5000` | held-out S1 (from LightGBM's set) and trained-on S1 in the evaluation |
| `FINETUNE_DIR` | `/kaggle/working/e5_finetune` | output folder |

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "experiments":     # run from the repo root, like the pipeline notebooks (no-op on Kaggle)
    os.chdir(Path.cwd().parent)

import collections
import csv
import gc
import glob
import importlib.util
import inspect
import json
import os
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

os.environ.setdefault("WANDB_DISABLED", "true")          # Kaggle ships wandb; the Trainer would otherwise ask for a key
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")  # the encoding processes below fork after tokenizers ran

for pkg in ("anyascii", "rapidfuzz", "datasets", "accelerate"):          # ISC, MIT, Apache 2.0, Apache 2.0
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import sentence_transformers  # noqa: E402
if int(sentence_transformers.__version__.split(".")[0]) < 3:     # the Trainer API below needs >= 3.0
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3.0"])
    raise SystemExit("sentence-transformers was upgraded: restart the kernel and run all again")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import pyarrow as pa  # noqa: E402
import pyarrow.compute as pc  # noqa: E402
import pyarrow.parquet as pq  # noqa: E402
import torch  # noqa: E402
from anyascii import anyascii  # noqa: E402  (ISC)
from rapidfuzz import fuzz  # noqa: E402  (MIT)
from rapidfuzz import process as rf_process  # noqa: E402
from sentence_transformers import SentenceTransformer  # noqa: E402  (Apache 2.0)

BASE_MODEL = os.environ.get("BASE_MODEL", "intfloat/multilingual-e5-small")
FINETUNE_COUNTRIES = [c.strip() for c in os.environ.get("FINETUNE_COUNTRIES", "India,US").split(",") if c.strip()]
MAX_PAIRS_PER_COUNTRY = int(os.environ.get("MAX_PAIRS_PER_COUNTRY", 500_000))
HARD_NEGATIVES = os.environ.get("HARD_NEGATIVES", "true") == "true"
HARD_NEG_TOP = int(os.environ.get("HARD_NEG_TOP", 10))
EPOCHS = float(os.environ.get("EPOCHS", 1))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", 256))
LEARNING_RATE = float(os.environ.get("LEARNING_RATE", 2e-5))
WARMUP_SHARE = 0.1
SCALE = float(os.environ.get("SCALE", 20))
LGBM_SEED = int(os.environ.get("LGBM_SEED", 42))                                  # 03 full's SEED
LGBM_TRAIN_S1_PER_COUNTRY = int(os.environ.get("LGBM_TRAIN_S1_PER_COUNTRY", 200_000))   # 03 full's TRAIN_S1_PER_COUNTRY
EVAL_COUNTRIES = [c.strip() for c in os.environ.get("EVAL_COUNTRIES", "India").split(",") if c.strip()]
EVAL_S1 = int(os.environ.get("EVAL_S1", 20_000))
SEEN_EVAL_S1 = int(os.environ.get("SEEN_EVAL_S1", 5_000))
EVAL_K = (5, 10, 30)         # not 1: an S1 has ~3.5 true records but only one can be first
SEED = 42
PREFIX = "query: "           # as 02 full: E5 expects "query: " for symmetric similarity
MAX_SEQ_LENGTH = 128         # as 02 full
OUT_DIR = Path(os.environ.get("FINETUNE_DIR") or ("/kaggle/working/e5_finetune" if os.path.isdir("/kaggle/working")
                                                  else "e5_finetune"))
MODEL_DIR = OUT_DIR / "e5-small-er"      # 02 full names its output folder after this: embeddings_full/e5-small-er/
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPUS = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if DEVICE == "cuda" else []
GPU_NAME = torch.cuda.get_device_name(0) if GPUS else "cpu"
# fp16 speeds up tensor-core GPUs (T4, A100) but is ~4x slower on GTX 16xx cards (as in 02 full)
USE_FP16 = bool(GPUS) and "GTX 16" not in GPU_NAME
ENCODE_BATCH = 1024 if USE_FP16 else 256
ENCODE_CHUNK = 1_000_000
SEARCH_DTYPE = torch.float16 if USE_FP16 else torch.float32
RESCORE_EXTRA = 32
GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory if GPUS else 0
TRAIN_GPUS = max(1, min(int(os.environ.get("TRAIN_GPUS", len(GPUS))), len(GPUS)))   # training processes, one per GPU
MINI_BATCH = int(os.environ.get("MINI_BATCH", 64))
CACHE_ABOVE = int(os.environ.get("CACHE_ABOVE", 128))   # rows per GPU above which the loss runs in MINI_BATCH chunks
FINETUNED_MODEL = os.environ.get("FINETUNED_MODEL", "")  # an already fine-tuned model folder: skip training, only evaluate
print(f"sentence-transformers {sentence_transformers.__version__} | {len(GPUS)} x {GPU_NAME} | fp16: {USE_FP16}")
print(f"fine-tune: {FINETUNE_COUNTRIES}, {MAX_PAIRS_PER_COUNTRY:,} pairs per country, hard negatives: {HARD_NEGATIVES} | "
      f"{EPOCHS:g} epoch(s), batch {BATCH_SIZE} on {TRAIN_GPUS} GPU(s), lr {LEARNING_RATE:g}, scale {SCALE:g} | "
      f"evaluation: {EVAL_COUNTRIES}")

## Inputs
`processed/` and the ground truth are found as in 02 full. 02 full's train buckets (for hard negatives) and word map
are found anywhere under the inputs; `BUCKET_DIR` and `WORDMAP_FILE` override the search.

In [ ]:
def find_processed_dir():
    """Folder with 01's per-country parquet: $PROCESSED_DIR, ./processed, /kaggle/working, any Kaggle input."""
    candidates = [Path(os.environ["PROCESSED_DIR"])] if os.environ.get("PROCESSED_DIR") else []
    candidates += [Path.cwd() / "processed", Path("/kaggle/working/processed")]
    candidates += sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/train_s1_*.parquet", recursive=True)})
    for c in candidates:
        if c.is_dir() and any(c.glob("train_s1_*.parquet")):
            return c
    raise FileNotFoundError("processed/ not found: attach 01's output (FULL_RUN=true) or set PROCESSED_DIR")


def find_ground_truth():
    """train_ground_truth.tsv under $DATA_ROOT, /kaggle/input or the local dataset folder; None if absent."""
    roots = [os.environ.get("DATA_ROOT", ""), "/kaggle/input", str(Path.cwd().parent / "Data" / "student_resource" / "dataset")]
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for h in glob.glob(os.path.join(root, "**", "train_ground_truth.tsv"), recursive=True)
                    if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    return None


def search_inputs(pattern):
    """Files matching pattern under /kaggle/input and /kaggle/working, base-model outputs first, then shortest path."""
    hits = [h for root in ("/kaggle/input", "/kaggle/working") if os.path.isdir(root)
            for h in glob.glob(os.path.join(root, pattern), recursive=True) if "__MACOSX" not in h]
    return sorted(set(hits), key=lambda h: ("multilingual-e5-small" not in h, len(h)))


PROCESSED_DIR = find_processed_dir()
GT_FILE = find_ground_truth()
assert GT_FILE is not None, "train_ground_truth.tsv not found: attach the competition data or set DATA_ROOT"
if os.environ.get("BUCKET_DIR"):
    BUCKET_DIR = Path(os.environ["BUCKET_DIR"])
else:
    _hits = [h for h in search_inputs("**/buckets/train_*.parquet") if "e5-small-er" not in h]
    BUCKET_DIR = Path(_hits[0]).parent if _hits else None
if os.environ.get("WORDMAP_FILE"):
    WORDMAP_FILE = Path(os.environ["WORDMAP_FILE"])
else:
    _hits = search_inputs("**/translit_wordmap.json")
    WORDMAP_FILE = Path(_hits[0]) if _hits else None
FILES = {(m.group(1), m.group(2)): f for f in sorted(PROCESSED_DIR.glob("train_s*_*.parquet"))
         if (m := re.match(r"^train_(s[123])_(.+)\.parquet$", f.name))}
TRAIN_COUNTRIES = sorted({c for s, c in FILES if s == "s1"})
print(f"processed: {PROCESSED_DIR}\nground truth: {GT_FILE}\nbuckets (hard negatives): {BUCKET_DIR}\n"
      f"word map: {WORDMAP_FILE or 'not found, learned below'}\ntrain countries: {TRAIN_COUNTRIES}\noutput: {OUT_DIR}")
assert set(FINETUNE_COUNTRIES) <= set(TRAIN_COUNTRIES), f"no train data for {set(FINETUNE_COUNTRIES) - set(TRAIN_COUNTRIES)}"
assert set(EVAL_COUNTRIES) <= set(TRAIN_COUNTRIES), f"no train data for {set(EVAL_COUNTRIES) - set(TRAIN_COUNTRIES)}"
if HARD_NEGATIVES and BUCKET_DIR is None:
    print("WARNING: no 02 full train buckets found: training with in-batch negatives only")


def country_files(country, sources):
    return [FILES[(s, country)] for s in sources if (s, country) in FILES]

## Text given to the model (copied from 02 full)
The functions below are copied unchanged from `02_full_e5_buckets.ipynb` (`TEXT_MODE=fixed`). If they change there,
change them here too: the model must be trained on the text it will embed.
* Latin-script records: `name_norm | addr_norm` from 01.
* Names or addresses with an Indian script: transliterated from the original text with anyascii and two fixes
  (Malayalam "റ്റ" → "tt", nasal dot → "n" before most consonants). Names also go through the word map
  ("praivet" → private).

`read_texts` also returns two flags per record: the evaluation uses them to report the Indian-script and
empty-address slices.

In [ ]:
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
RE_NATIVE = re.compile("[^" + "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES) + "]")
_INDIC_BLOCKS = range(0x0900, 0x0D80, 0x80)                     # Devanagari, Bengali, ..., Malayalam (same layout)
_ANUSVARA = "".join(chr(b + 0x02) for b in _INDIC_BLOCKS)
_LABIALS = "".join(chr(b + o) for b in _INDIC_BLOCKS for o in range(0x2A, 0x2F))      # pa, pha, ba, bha, ma
RE_ANUSVARA_N = re.compile(f"[{_ANUSVARA}](?=[ऀ-ൿ])(?![{_LABIALS}])")


def translit(s):
    """anyascii with the two fixes above."""
    return anyascii(RE_ANUSVARA_N.sub("n", s.replace("റ്റ", "ട്ട")))


def latin(raw, cleaned):
    """cleaned (01's text) for Latin-script text, else the transliterated original."""
    return " ".join(translit(raw).lower().split()) if RE_NATIVE.search(raw) else cleaned


WORDMAP_MIN_PAIRS, WORDMAP_MIN_SHARE, WORDMAP_MIN_RATIO = 20, 0.6, 55
_RE_WORD2 = re.compile(r"[a-z]{2,}")


def learn_word_map(processed_dir, gt_file):
    """{transliterated word: English word} from the train true pairs of every country (only Indian-script names count)."""
    gt = pd.read_csv(gt_file, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
    gp = gt[gt["matched_entity_ids"] != ""]
    gp = gp.assign(cand=gp["matched_entity_ids"].str.split(",")).explode("cand")
    owner = dict(zip(gp["cand"], gp["source1_entity_id"]))
    del gt, gp
    s1_name = {}
    for f in sorted(Path(processed_dir).glob("train_s1_*.parquet")):
        d = pd.read_parquet(f, columns=["entity_id", "business_name"])
        s1_name.update(zip(d["entity_id"], d["business_name"].fillna("").astype(str)))
    pair_counts, tok_counts = collections.Counter(), collections.Counter()
    for f in sorted(list(Path(processed_dir).glob("train_s2_*.parquet")) + list(Path(processed_dir).glob("train_s3_*.parquet"))):
        d = pd.read_parquet(f, columns=["entity_id", "business_name"])
        for e, name in zip(d["entity_id"], d["business_name"].fillna("").astype(str)):
            s = owner.get(e)
            if s is None or s not in s1_name or not RE_NATIVE.search(name):
                continue
            a = set(_RE_WORD2.findall(s1_name[s].lower()))
            for t in set(_RE_WORD2.findall(translit(name).lower())):
                tok_counts[t] += 1
                if t in a or not a:
                    continue
                best = rf_process.extractOne(t, a, scorer=fuzz.ratio, score_cutoff=WORDMAP_MIN_RATIO)
                if best:
                    pair_counts[(t, best[0])] += 1
    best_of = {}
    for (t, e), n in pair_counts.items():
        if n > best_of.get(t, (0, ""))[0] or (n == best_of[t][0] and e < best_of[t][1]):
            best_of[t] = (n, e)
    return {t: e for t, (n, e) in sorted(best_of.items())
            if n >= WORDMAP_MIN_PAIRS and n / tok_counts[t] >= WORDMAP_MIN_SHARE}


if WORDMAP_FILE is not None:
    WORDMAP = json.loads(WORDMAP_FILE.read_text(encoding="utf-8"))
else:
    t0 = time.time()
    WORDMAP = learn_word_map(PROCESSED_DIR, GT_FILE)
    print(f"word map learned from the train pairs in {time.time() - t0:.0f}s")
(OUT_DIR / "translit_wordmap.json").write_text(json.dumps(WORDMAP, ensure_ascii=False, indent=0, sort_keys=True),
                                               encoding="utf-8")
print(f"word map: {len(WORDMAP)} words, e.g. " + ", ".join(f"{k} -> {v}" for k, v in list(WORDMAP.items())[:12]))


def translit_name(s):
    """Transliterated Indian-script name with the word map applied ('प्राइवेट लिमिटेड' -> 'private limited')."""
    return " ".join(WORDMAP.get(w, w) for w in translit(s).lower().split())


def latin_name(raw, cleaned):
    """latin() for names: Indian-script names transliterated and word-mapped."""
    return translit_name(raw) if RE_NATIVE.search(raw) else cleaned


TEXT_COLS = ["name_norm", "addr_norm", "business_name", "business_address"]


def read_texts(paths):
    """entity_ids, e5 texts (02 full, TEXT_MODE=fixed), and per record: Indian-script name, empty address."""
    df = pd.concat([pd.read_parquet(p, columns=["entity_id"] + TEXT_COLS) for p in paths], ignore_index=True)
    for c in TEXT_COLS:
        df[c] = df[c].fillna("").astype(str)
    names = [latin_name(r, c) for r, c in zip(df["business_name"], df["name_norm"])]
    addrs = [latin(r, c) for r, c in zip(df["business_address"], df["addr_norm"])]
    texts = [f"{n.strip()} | {a.strip()}" if a.strip() else n.strip() for n, a in zip(names, addrs)]
    native = np.fromiter((RE_NATIVE.search(x) is not None for x in df["business_name"]), bool, len(df))
    empty = np.fromiter((not a.strip() for a in addrs), bool, len(df))
    return df["entity_id"].astype(str).to_numpy(dtype=object), texts, native, empty

## Positive pairs, and the S1 kept away from fine-tuning
03 full draws its LightGBM training S1 like this: for each train country in sorted order, it takes
`rng.choice(s1_all, TRAIN_S1_PER_COUNTRY, replace=False)` with `rng = np.random.default_rng(SEED)`. Here `s1_all` is
the S1 in the order of its train bucket file. 02 full writes the buckets grouped by S1 in the order of 01's
`train_s1_<country>.parquet` (`np.lexsort((-score, s1_index))`), so that order does not depend on the embedding
model, and the same draw is repeated below. When 02 full's buckets are attached, their S1 order is checked against
the processed file.

Only S1 **outside** that set are fine-tuned on. The evaluation's held-out S1 come from inside it: they are exactly
the S1 LightGBM will learn from.

In [ ]:
gt = pd.read_csv(GT_FILE, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
gp = gt[gt["matched_entity_ids"] != ""]
PAIRS = (gp.assign(cand=gp["matched_entity_ids"].str.split(",")).explode("cand")
         .rename(columns={"source1_entity_id": "s1"})[["s1", "cand"]].reset_index(drop=True))
OWNER = pd.Series(PAIRS["s1"].to_numpy(), index=PAIRS["cand"].to_numpy())      # S2/S3 record -> its true S1
OWNER = OWNER[~OWNER.index.duplicated()]         # one owner per record (EDA: none is reused); guards .map below
del gt, gp
print(f"{len(PAIRS):,} true (S1, S2/S3) pairs in the ground truth")

S1_ORDER = {c: pd.read_parquet(FILES[("s1", c)], columns=["entity_id"])["entity_id"].astype(str).to_numpy(dtype=object)
            for c in TRAIN_COUNTRIES}
for c in TRAIN_COUNTRIES:
    path = BUCKET_DIR / f"train_{c}.parquet" if BUCKET_DIR else None
    if path is not None and path.exists():
        in_buckets = pd.read_parquet(path, columns=["s1_entity_id"])["s1_entity_id"].unique()
        assert len(in_buckets) == len(S1_ORDER[c]) and (in_buckets == S1_ORDER[c]).all(), \
            f"{c}: the S1 order of 02 full's buckets differs from processed/: 03's LightGBM S1 cannot be reproduced"
        print(f"{c}: S1 order of 02 full's train buckets matches processed/ ({len(in_buckets):,} S1)")

_rng03 = np.random.default_rng(LGBM_SEED)
LGBM_S1 = {c: set(_rng03.choice(S1_ORDER[c], min(LGBM_TRAIN_S1_PER_COUNTRY, len(S1_ORDER[c])), replace=False))
           for c in TRAIN_COUNTRIES}
(OUT_DIR / "lgbm_train_s1.txt").write_text("\n".join(sorted(s for v in LGBM_S1.values() for s in v)), encoding="utf-8")
summary = []
for c in TRAIN_COUNTRIES:
    p = PAIRS[PAIRS["s1"].isin(set(S1_ORDER[c]))]
    summary.append({"country": c, "train S1": len(S1_ORDER[c]), "LightGBM S1 (never fine-tuned on)": len(LGBM_S1[c]),
                    "true pairs": len(p), "pairs available for fine-tuning": int((~p["s1"].isin(LGBM_S1[c])).sum())})
pd.DataFrame(summary).set_index("country")

## Training rows
For each country: up to `MAX_PAIRS_PER_COUNTRY` random true pairs whose S1 is outside LightGBM's set, as
`(anchor = S1 text, positive = true record's text, negative = a lookalike's text)`.

The hard negative comes from the S1's bucket in 02 full's train buckets. It is drawn at random among the S1's
`HARD_NEG_TOP` best-scored candidates that are not its true matches: the records it competes with at test time.
A negative whose text equals the anchor's or the positive's is skipped. An S1 with no usable lookalike gets a
random record of its country.

**Leak check**, before any training:
* The notebook stops if a training pair involves one of LightGBM's S1 (the evaluation's "unseen" S1) or one of their
  true records. `pairs touching evaluation S1` must be 0.
* It also counts `evaluation pairs with a trained pair's exact texts`: evaluation pairs whose two texts are exactly
  those of a training pair (other entities with identical text, e.g. a business listed twice). The model has in
  effect seen those. Expect 0 or a handful.

In [ ]:
def hard_negative_lists(country, s1_ids):
    """{S1: its HARD_NEG_TOP best-scored bucket candidates that are not its true matches}, from 02 full's train buckets."""
    path = BUCKET_DIR / f"train_{country}.parquet" if BUCKET_DIR else None
    if not HARD_NEGATIVES or path is None or not path.exists():
        return {}
    cols = [c for c in ("s1_entity_id", "candidate_entity_id", "score") if c in pq.read_schema(path).names]
    t = pq.read_table(path, columns=cols)
    keep = pc.is_in(t["s1_entity_id"].cast(pa.large_string()), value_set=pa.array(sorted(s1_ids), pa.large_string()))
    b = t.filter(keep).to_pandas()
    del t, keep
    b = b[b["candidate_entity_id"].map(OWNER).to_numpy() != b["s1_entity_id"].to_numpy()]   # drop the true matches
    if "score" in b:
        b = b.sort_values(["s1_entity_id", "score"], ascending=[True, False], kind="stable")
    b = b.groupby("s1_entity_id", sort=False).head(HARD_NEG_TOP)
    return {s: g.to_numpy(dtype=object) for s, g in b.groupby("s1_entity_id", sort=False)["candidate_entity_id"]}


rng = np.random.default_rng(SEED)
TRAIN, USED_PAIRS, STATS = {}, {}, []
for country in FINETUNE_COUNTRIES:
    t0 = time.time()
    ids, texts, _, _ = read_texts(country_files(country, ("s1", "s2", "s3")))
    text_of = dict(zip(ids, texts))
    pool = np.array([i for i in ids if not i.startswith("S1-")], dtype=object)      # the country's S2/S3 records
    del ids, texts
    p = PAIRS[PAIRS["s1"].isin(set(S1_ORDER[country]) - LGBM_S1[country])]
    anchor = p["s1"].map(text_of)
    positive = p["cand"].map(text_of)
    p = p[anchor.notna().to_numpy() & positive.notna().to_numpy() & (anchor != "").to_numpy() & (positive != "").to_numpy()]
    if len(p) > MAX_PAIRS_PER_COUNTRY:
        p = p.iloc[np.sort(rng.choice(len(p), MAX_PAIRS_PER_COUNTRY, replace=False))]
    p = p.iloc[rng.permutation(len(p))].reset_index(drop=True)
    anc = p["s1"].map(text_of).to_numpy(dtype=object)
    pos = p["cand"].map(text_of).to_numpy(dtype=object)
    lists = hard_negative_lists(country, set(p["s1"]))
    neg, from_bucket = np.empty(len(p), dtype=object), 0
    for i, s in enumerate(p["s1"].to_numpy()):
        opts, chosen = lists.get(s), None
        if opts is not None and len(opts):
            for _ in range(3):
                t = text_of.get(opts[rng.integers(len(opts))], "")
                if t and t != anc[i] and t != pos[i]:
                    chosen, from_bucket = t, from_bucket + 1
                    break
        while chosen is None:
            t = text_of.get(pool[rng.integers(len(pool))], "")
            if t and t != anc[i] and t != pos[i]:
                chosen = t
        neg[i] = chosen
    # leak check, before any training: no evaluation S1 (LightGBM's) and none of their true records in these pairs
    ev = PAIRS[PAIRS["s1"].isin(LGBM_S1[country])]
    touching = int(p["s1"].isin(LGBM_S1[country]).sum() + p["cand"].isin(set(ev["cand"])).sum())
    assert touching == 0, f"{country}: {touching} training pairs involve an evaluation S1 or one of its true records"
    trained_texts = set(zip(anc, pos))
    same_texts = sum((a, b) in trained_texts for a, b in zip(ev["s1"].map(text_of).fillna(""), ev["cand"].map(text_of).fillna("")))
    frame = pd.DataFrame({"anchor": [PREFIX + t for t in anc], "positive": [PREFIX + t for t in pos]})
    if lists:                                   # without buckets: in-batch negatives only (a random extra one adds little)
        frame["negative"] = [PREFIX + t for t in neg]
    TRAIN[country] = frame
    USED_PAIRS[country] = set(p["s1"] + "|" + p["cand"])
    STATS.append({"country": country, "training rows": len(frame), "S1": p["s1"].nunique(),
                  "hard negative from the bucket": round(from_bucket / max(len(p), 1), 3) if lists else 0.0,
                  "pairs touching evaluation S1": touching, "evaluation pairs with a trained pair's exact texts": same_texts,
                  "seconds": round(time.time() - t0)})
    print(f"{country}: {len(frame):,} rows in {time.time() - t0:.0f}s, e.g.")
    for r in frame.head(3).itertuples(index=False):
        print("   ", " || ".join(x[len(PREFIX):][:70] for x in r))
    del text_of, pool, lists, p, anc, pos, neg, frame
    gc.collect()
(OUT_DIR / "finetune_s1.txt").write_text("\n".join(sorted({k.split("|")[0] for v in USED_PAIRS.values() for k in v})),
                                         encoding="utf-8")
pd.DataFrame(STATS).set_index("country")

## Fine-tuning
* **Loss:** `MultipleNegativesRankingLoss`. Cross-entropy over the similarities of each anchor to its positive, to
  every other positive and negative of the batch, and to its own hard negative, with cosine × `SCALE` as logits.
* **Both GPUs (DDP):** the training runs as a separate script, `train_e5.py`, with one process per GPU (`torchrun`).
  * Each GPU takes `BATCH_SIZE / 2` rows per step. The embeddings are gathered across the GPUs, so every anchor
    still competes with all `BATCH_SIZE` rows of the step, as on one GPU. The script's first line says whether
    this sentence-transformers version can gather.
  * If the 2-GPU run fails, the notebook trains again on one GPU.
* **Memory:** above 128 rows per GPU, the loss becomes `CachedMultipleNegativesRankingLoss`. It is the same loss
  with the same gradients, computed `MINI_BATCH` rows at a time. With the plain loss, a T4 ran out of memory on 256
  rows × 3 texts per step (14.6 GB).
* `NO_DUPLICATES`: a text never appears twice in a batch. Two matches of one S1 are then never in the same batch,
  where one would be counted as the other's negative.
* `PROPORTIONAL`: each batch comes from one country, in proportion to its rows, so in-batch negatives come from the
  same country, like the records an S1 competes with.
* The script prints the step, the loss and the time left every 50 steps. The loss should fall quickly, then
  flatten.
* **Already trained?** Set `FINETUNED_MODEL` to the model folder of an earlier run: training is skipped, the model is
  copied to this run's output and evaluated. Keep the other settings as in that run, so the "seen" S1 of the
  evaluation are the ones it trained on.

In [ ]:
TRAIN_SCRIPT = r'''
"""02b's fine-tuning, started by the notebook: one process per GPU under torchrun, or a single process."""
import inspect
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from transformers import TrainerCallback
from transformers.trainer_callback import PrinterCallback
try:                                             # sentence-transformers >= 6 moved these; older versions keep the old path
    from sentence_transformers.sentence_transformer import losses
    from sentence_transformers.sentence_transformer.training_args import BatchSamplers, MultiDatasetBatchSamplers
except ImportError:
    from sentence_transformers import losses
    from sentence_transformers.training_args import BatchSamplers, MultiDatasetBatchSamplers

cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
world = int(os.environ.get("WORLD_SIZE", 1))
rank = int(os.environ.get("LOCAL_RANK", 0))
per_device = cfg["batch_size"] // world
model = SentenceTransformer(cfg["base_model"], device=f"cuda:{rank}" if torch.cuda.is_available() else "cpu")
model.max_seq_length = cfg["max_seq_length"]
data = {c: Dataset.from_pandas(pd.read_parquet(p), preserve_index=False) for c, p in cfg["train_files"].items()}
train_data = next(iter(data.values())) if len(data) == 1 else data

cached = per_device > cfg["cache_above"]
loss_cls = losses.CachedMultipleNegativesRankingLoss if cached else losses.MultipleNegativesRankingLoss
kwargs = {"scale": cfg["scale"]}
if cached:
    kwargs["mini_batch_size"] = cfg["mini_batch"]
gather = world > 1 and "gather_across_devices" in inspect.signature(loss_cls.__init__).parameters
if gather:
    kwargs["gather_across_devices"] = True
loss = loss_cls(model, **kwargs)

args = SentenceTransformerTrainingArguments(
    output_dir=cfg["checkpoint_dir"],
    num_train_epochs=cfg["epochs"],
    per_device_train_batch_size=per_device,
    learning_rate=cfg["learning_rate"],
    warmup_steps=cfg["warmup_steps"],
    fp16=cfg["fp16"],
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
    dataloader_drop_last=True,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    seed=cfg["seed"],
    disable_tqdm=True,
    ddp_timeout=600,                             # a stuck GPU link fails after 10 min instead of 30
)


class Progress(TrainerCallback):
    """Step, loss and time left at every log (first process only)."""

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_world_process_zero and logs and "loss" in logs:
            minutes = (time.time() - self.t0) / 60
            left = minutes / max(state.global_step, 1) * (state.max_steps - state.global_step)
            print(f"step {state.global_step:,}/{state.max_steps:,}  loss {logs['loss']:.4f}  "
                  f"lr {logs.get('learning_rate', 0):.2e}  {minutes:.1f} min, ~{left:.0f} min left", flush=True)


trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_data, loss=loss, callbacks=[Progress()])
trainer.remove_callback(PrinterCallback)
if trainer.is_world_process_zero():
    chunks = f" in chunks of {cfg['mini_batch']} rows" if cached else ""
    negatives = " gathered across GPUs" if gather else (" per GPU only (this version cannot gather)" if world > 1 else "")
    print(f"{world} process(es) x {per_device} rows per step | {loss_cls.__name__}{chunks} | "
          f"in-batch negatives{negatives}", flush=True)
t0 = time.time()
trainer.train()
if trainer.is_world_process_zero():
    model.save(cfg["model_dir"])
    log = pd.DataFrame([r for r in trainer.state.log_history if "loss" in r])
    (log if len(log) else pd.DataFrame(columns=["step", "loss", "learning_rate"])).to_csv(cfg["log_file"], index=False)
    Path(cfg["done_file"]).write_text(json.dumps({
        "minutes": round((time.time() - t0) / 60, 1), "processes": world, "rows_per_gpu": per_device,
        "loss": loss_cls.__name__, "mini_batch": cfg["mini_batch"] if cached else None,
        "gather_across_gpus": gather}), encoding="utf-8")
if torch.distributed.is_available() and torch.distributed.is_initialized():
    torch.distributed.barrier()                  # the other processes wait until the model is saved
    torch.distributed.destroy_process_group()
'''

SCRIPT, RUN_FILE, DONE_FILE = OUT_DIR / "train_e5.py", OUT_DIR / "train_run.json", OUT_DIR / "train_done.json"
TRAIN_DIR = OUT_DIR / "train_rows"               # the training rows, for the training processes (deleted afterwards)


def launch(n_gpus):
    """Run the training script on n_gpus GPUs (torchrun for 2 or more), streaming its output; returns the exit code."""
    DONE_FILE.unlink(missing_ok=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    env.setdefault("NCCL_P2P_DISABLE", "1")      # GPU-to-GPU traffic through the host: direct copies hang on some cloud machines
    # less GPU memory fragmentation in training (PyTorch >= 2.9 reads the first name, older versions the second).
    # Training processes only: in this notebook it breaks the 2-GPU search of the evaluation ("CUDA driver error:
    # operation not permitted")
    for name in ("PYTORCH_ALLOC_CONF", "PYTORCH_CUDA_ALLOC_CONF"):
        env.setdefault(name, "expandable_segments:True")
    if n_gpus > 1:
        cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={n_gpus}",
               str(SCRIPT), str(RUN_FILE)]
    else:
        cmd = [sys.executable, str(SCRIPT), str(RUN_FILE)]
        if GPUS:                                 # one visible GPU, or the Trainer would wrap the model in DataParallel
            env["CUDA_VISIBLE_DEVICES"] = env.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env,
                            encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


if FINETUNED_MODEL:
    assert (Path(FINETUNED_MODEL) / "modules.json").exists(), f"FINETUNED_MODEL={FINETUNED_MODEL} is not a model folder"
    if Path(FINETUNED_MODEL).resolve() != MODEL_DIR.resolve():
        shutil.copytree(FINETUNED_MODEL, MODEL_DIR, dirs_exist_ok=True)
    TRAINED, TRAIN_MINUTES = {"reused": FINETUNED_MODEL}, 0.0
    earlier_log = Path(FINETUNED_MODEL).parent / "train_log.csv"
    TRAIN_LOG = pd.read_csv(earlier_log) if earlier_log.exists() else pd.DataFrame(columns=["step", "loss", "learning_rate"])
    if earlier_log.exists():
        TRAIN_LOG.to_csv(OUT_DIR / "train_log.csv", index=False)
    print(f"training skipped: FINETUNED_MODEL={FINETUNED_MODEL}, copied to {MODEL_DIR}")
else:
    TRAIN_DIR.mkdir(exist_ok=True)
    TRAIN_FILES = {}
    for c, df in TRAIN.items():
        TRAIN_FILES[c] = str(TRAIN_DIR / f"{c}.parquet")
        df.to_parquet(TRAIN_FILES[c], index=False)
    steps = int(sum(len(df) // BATCH_SIZE for df in TRAIN.values()) * EPOCHS)
    SCRIPT.write_text(TRAIN_SCRIPT, encoding="utf-8")
    RUN_FILE.write_text(json.dumps({
        "base_model": BASE_MODEL, "train_files": TRAIN_FILES, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE, "warmup_steps": int(WARMUP_SHARE * steps), "scale": SCALE, "seed": SEED,
        "fp16": USE_FP16, "max_seq_length": MAX_SEQ_LENGTH, "mini_batch": MINI_BATCH, "cache_above": CACHE_ABOVE,
        "checkpoint_dir": str(OUT_DIR / "checkpoints"), "model_dir": str(MODEL_DIR),
        "log_file": str(OUT_DIR / "train_log.csv"), "done_file": str(DONE_FILE)}), encoding="utf-8")
    _ = SentenceTransformer(BASE_MODEL, device="cpu")    # download once here, not in every training process
    del _
    print(f"{sum(len(df) for df in TRAIN.values()):,} rows, ~{steps:,} steps of {BATCH_SIZE} rows on {TRAIN_GPUS} GPU(s), "
          f"warm-up {int(WARMUP_SHARE * steps)} steps", flush=True)
    code = launch(TRAIN_GPUS)
    if code != 0 and TRAIN_GPUS > 1:
        print(f"\ntraining on {TRAIN_GPUS} GPUs failed (exit code {code}): training again on one GPU\n", flush=True)
        code = launch(1)
    assert code == 0 and DONE_FILE.exists(), f"training failed (exit code {code}): see the output above"
    TRAINED = json.loads(DONE_FILE.read_text(encoding="utf-8"))
    TRAIN_MINUTES = TRAINED["minutes"]
    TRAIN_LOG = pd.read_csv(OUT_DIR / "train_log.csv")
    shutil.rmtree(TRAIN_DIR, ignore_errors=True)
    print(f"trained in {TRAIN_MINUTES} min ({TRAINED}); model saved to {MODEL_DIR}")
gc.collect()
TRAIN_LOG[["step", "loss", "learning_rate"]].iloc[np.unique(np.linspace(0, len(TRAIN_LOG) - 1, 12).astype(int))] \
    if len(TRAIN_LOG) else "no loss logged"

## Evaluation at full size: base vs fine-tuned e5 recall
For each country in `EVAL_COUNTRIES`, both models encode **every** train S2/S3 record of the country, as in 02 full.
The search is exact, over mean-centered vectors, with the same code as 02 full. Two groups of S1 are measured:
* **unseen:** `EVAL_S1` S1 with at least one true match, from LightGBM's set. The model never trained on them, so
  this is the number that should carry over to test.
* **seen:** `SEEN_EVAL_S1` S1 whose pairs were used for fine-tuning, counting only those pairs. How far this
  exceeds "unseen" shows how much the model memorized.

Recall@k is the share of true pairs whose record is among the S1's k best (k = 5, 10, 30; not 1, since an S1 has
~3.5 true records and only one can be first). It is reported for all pairs and by the kind of true record:
Indian-script name, empty address, and Latin-script with an address.

Sanity check: the base model's unseen recall@30 should be close to 02 full's train report for e5 top-30 (India
0.9471, US 0.9833). The extra searches of 02 full come on top of this in the real buckets.

In [ ]:
def load_encoder(name):
    """The model on the CPU when it is copied to several GPUs (as in 02 full), else on the GPU; fp16 on T4s."""
    m = SentenceTransformer(str(name), device="cpu" if len(GPUS) > 1 else (GPUS[0] if GPUS else "cpu"))
    m.max_seq_length = MAX_SEQ_LENGTH
    if USE_FP16:
        m.half()
    return m


def embed_all(model, texts, label):
    """Unit-norm float16 embeddings aligned with texts: each unique text encoded once, longest first, on all GPUs."""
    codes, uniq = pd.factorize(np.asarray(texts, dtype=object))
    order = np.argsort(-np.fromiter((len(t) for t in uniq), np.int64, len(uniq)), kind="stable")
    pool = model.start_multi_process_pool(target_devices=GPUS) if len(GPUS) > 1 else None
    out, t0 = None, time.time()
    try:
        for s in range(0, len(order), ENCODE_CHUNK):
            idx = order[s:s + ENCODE_CHUNK]
            batch = [PREFIX + t for t in uniq[idx]]
            if pool is None:
                vecs = model.encode(batch, batch_size=ENCODE_BATCH, convert_to_numpy=True, show_progress_bar=False)
            elif "pool" in inspect.signature(model.encode).parameters:           # sentence-transformers >= 5
                vecs = model.encode(batch, pool=pool, batch_size=ENCODE_BATCH, chunk_size=16 * ENCODE_BATCH,
                                    convert_to_numpy=True, show_progress_bar=False)
            else:
                vecs = model.encode_multi_process(batch, pool, batch_size=ENCODE_BATCH, chunk_size=16 * ENCODE_BATCH)
            vecs = np.asarray(vecs, dtype=np.float32)
            vecs /= np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-12)
            if out is None:
                out = np.empty((len(uniq), vecs.shape[1]), np.float16)
            out[idx] = vecs.astype(np.float16)
        print(f"  {label}: {len(uniq):,} unique texts, {len(uniq) / max(time.time() - t0, 1e-9):,.0f} texts/s")
    finally:
        if pool is not None:
            SentenceTransformer.stop_multi_process_pool(pool)
    return out[codes]


def _normalize(x):
    return x / x.norm(dim=1, keepdim=True).clamp_min(1e-12)


def exact_search(base16, queries16, mu, k):
    """Top-k rows of base16 for every row of queries16, by inner product of mean-centered, re-normalized vectors
    (02 full's search)."""
    n, d = base16.shape
    scores = np.empty((len(queries16), k), np.float32)
    hits = np.empty((len(queries16), k), np.int64)
    if not GPUS:
        base = base16.astype(np.float32) - mu
        base /= np.maximum(np.linalg.norm(base, axis=1, keepdims=True), 1e-12)
        for s in range(0, len(queries16), 1024):
            q = queries16[s:s + 1024].astype(np.float32) - mu
            q /= np.maximum(np.linalg.norm(q, axis=1, keepdims=True), 1e-12)
            sims = q @ base.T
            top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
            order = np.argsort(-np.take_along_axis(sims, top, axis=1), axis=1)
            hits[s:s + 1024] = np.take_along_axis(top, order, axis=1)
            scores[s:s + 1024] = np.take_along_axis(sims, hits[s:s + 1024], axis=1)
        return scores, hits
    shortlist = min(n, k + RESCORE_EXTRA)
    elem = 2 if SEARCH_DTYPE == torch.float16 else 4
    rows = int(np.clip((GPU_MEMORY // 8) // (n * elem), 64, 4096))    # query rows per block: score block <= 1/8 of GPU memory
    blocks = [(s, min(s + rows, len(queries16))) for s in range(0, len(queries16), rows)]

    def run(device, my_blocks):
        with torch.cuda.device(device):
            mu_t = torch.from_numpy(mu).to(device)
            base = torch.empty((n, d), dtype=SEARCH_DTYPE, device=device)
            step = max(100_000, 384_000_000 // d)
            for s in range(0, n, step):
                chunk = torch.from_numpy(base16[s:s + step]).to(device).float() - mu_t
                base[s:s + len(chunk)] = _normalize(chunk).to(SEARCH_DTYPE)
            del chunk
            for s, e in my_blocks:
                q = _normalize(torch.from_numpy(queries16[s:e]).to(device).float() - mu_t)
                cand = torch.topk(q.to(SEARCH_DTYPE) @ base.T, shortlist, dim=1).indices
                rescored = torch.einsum("qd,qkd->qk", q, base[cand].float())
                top = torch.topk(rescored, k, dim=1)
                scores[s:e] = top.values.cpu().numpy()
                hits[s:e] = torch.gather(cand, 1, top.indices).cpu().numpy()
            del base
            torch.cuda.empty_cache()

    with ThreadPoolExecutor(len(GPUS)) as ex:
        list(ex.map(run, GPUS, [blocks[i::len(GPUS)] for i in range(len(GPUS))]))
    return scores, hits

In [ ]:
rng_eval = np.random.default_rng(SEED + 1)
EVAL_ROWS = []
for country in EVAL_COUNTRIES:
    t_c = time.time()
    s_ids, s_texts, _, _ = read_texts(country_files(country, ("s1",)))
    c_ids, c_texts, c_native, c_empty = read_texts(country_files(country, ("s2", "s3")))
    s_pos = pd.Series(np.arange(len(s_ids)), index=s_ids)
    c_pos = pd.Series(np.arange(len(c_ids)), index=c_ids)
    p = PAIRS[PAIRS["s1"].isin(s_pos.index) & PAIRS["cand"].isin(c_pos.index)]
    with_match = set(p["s1"])
    unseen = np.array(sorted(s for s in LGBM_S1[country] if s in with_match), dtype=object)
    groups = {"unseen": unseen[rng_eval.permutation(len(unseen))[:EVAL_S1]]}
    used = USED_PAIRS.get(country, set())
    if used:
        seen = np.array(sorted({k.split("|")[0] for k in used}), dtype=object)
        groups["seen"] = seen[rng_eval.permutation(len(seen))[:SEEN_EVAL_S1]]
    q_ids = np.concatenate(list(groups.values()))
    q_group = np.concatenate([[g] * len(v) for g, v in groups.items()])
    tp = p[p["s1"].isin(set(q_ids))].reset_index(drop=True)
    q_index = pd.Series(np.arange(len(q_ids)), index=q_ids)
    tq = q_index.reindex(tp["s1"]).to_numpy().astype(np.int64)
    tc = c_pos.reindex(tp["cand"]).to_numpy().astype(np.int64)
    t_group = q_group[tq]
    if "seen" in groups:                                   # seen: only the pairs actually used in training
        was_used = (tp["s1"] + "|" + tp["cand"]).isin(used).to_numpy()
        keep = (t_group == "unseen") | was_used
        tq, tc, t_group = tq[keep], tc[keep], t_group[keep]
    slices = {"all": np.ones(len(tc), bool), "Indian-script name": c_native[tc], "empty address": c_empty[tc],
              "Latin, with address": ~c_native[tc] & ~c_empty[tc]}
    q_texts = [s_texts[i] for i in s_pos.reindex(q_ids).to_numpy()]
    print(f"{country}: {len(c_ids):,} S2/S3 records searched; {len(q_ids):,} S1 "
          f"({', '.join(f'{g} {len(v):,}' for g, v in groups.items())}); {len(tc):,} true pairs")
    for label, name in (("base", BASE_MODEL), ("fine-tuned", MODEL_DIR)):
        enc = load_encoder(name)
        c_emb = embed_all(enc, c_texts, f"{country} {label} S2/S3")
        q_emb = embed_all(enc, q_texts, f"{country} {label} S1")
        del enc
        gc.collect()
        mu = c_emb.mean(axis=0, dtype=np.float64).astype(np.float32)[None, :]
        _, hits = exact_search(c_emb, q_emb, mu, max(EVAL_K))
        match = hits[tq] == tc[:, None]
        found_rank = np.where(match.any(axis=1), match.argmax(axis=1), max(EVAL_K))
        for g in groups:
            for s_name, s_mask in slices.items():
                m = (t_group == g) & s_mask
                if m.any():
                    EVAL_ROWS.append({"country": country, "S1": g, "true record": s_name, "model": label,
                                      "true pairs": int(m.sum()),
                                      **{f"recall@{k}": round(float((found_rank[m] < k).mean()), 4) for k in EVAL_K}})
        del c_emb, q_emb, hits, match
        gc.collect()
        if GPUS:
            torch.cuda.empty_cache()
    print(f"{country}: evaluated in {(time.time() - t_c) / 60:.1f} min")

EVAL = pd.DataFrame(EVAL_ROWS)
EVAL.to_csv(OUT_DIR / "eval_recall.csv", index=False)
wide = EVAL.pivot_table(index=["country", "S1", "true record", "true pairs"], columns="model",
                        values=[f"recall@{k}" for k in EVAL_K])
wide = wide.reindex(pd.MultiIndex.from_frame(EVAL[["country", "S1", "true record", "true pairs"]].drop_duplicates()))
for k in EVAL_K:
    wide[(f"recall@{k}", "gain")] = wide[(f"recall@{k}", "fine-tuned")] - wide[(f"recall@{k}", "base")]
wide = wide[[(f"recall@{k}", m) for k in EVAL_K for m in ("base", "fine-tuned", "gain")]].round(4)
wide

## Reading the results, and what to do next
* **"unseen, all, recall@30" is the number that matters.** It is the share of true pairs the model puts in the
  top 30 for S1 it never trained on. 02 full's buckets add the extra searches on top of it, and 03 full judges
  those buckets.
* **Small or negative gain:** keep the base model. Fine-tuning then only adds risk (France, memorization).
* **Clear gain** (say +0.005 or more at recall@30 on India): run 02 full with the new model, then 03 full, as in
  the first cell. Compare 03's cross-validation with 0.9796 and 02's train report with the base buckets. The e5
  score is the input of the matcher's strongest features, so the end result can move more or less than recall.
* **"seen" far above "unseen"** (e.g. 0.99 vs 0.95): the model memorized the training pairs. Their S1 still
  compete with LightGBM's S1 in the train buckets, so 03's cross-validation may then be slightly optimistic. Use
  fewer pairs or a lower learning rate.
* **France:** no label can check it. The cross-country run (`FINETUNE_COUNTRIES=India`, `EVAL_COUNTRIES=US`) is
  the only proxy. If fine-tuning on India hurts the US, expect it to hurt France too.

In [ ]:
CONFIG = {"base_model": BASE_MODEL, "model_dir": str(MODEL_DIR), "finetune_countries": FINETUNE_COUNTRIES,
          "max_pairs_per_country": MAX_PAIRS_PER_COUNTRY, "hard_negatives": HARD_NEGATIVES and BUCKET_DIR is not None,
          "hard_neg_top": HARD_NEG_TOP, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
          "warmup_share": WARMUP_SHARE, "scale": SCALE, "prefix": PREFIX, "max_seq_length": MAX_SEQ_LENGTH,
          "lgbm_seed": LGBM_SEED, "lgbm_train_s1_per_country": LGBM_TRAIN_S1_PER_COUNTRY, "training": STATS,
          "train_minutes": TRAIN_MINUTES, "training_run": TRAINED, "word_map_words": len(WORDMAP), "processed_dir": str(PROCESSED_DIR),
          "bucket_dir": str(BUCKET_DIR), "sentence_transformers": sentence_transformers.__version__,
          "eval": EVAL_ROWS}
(OUT_DIR / "finetune_config.json").write_text(json.dumps(CONFIG, indent=1, default=str), encoding="utf-8")
shutil.rmtree(OUT_DIR / "checkpoints", ignore_errors=True)    # save_strategy="no": only Trainer scaffolding in there
for f in (RUN_FILE, DONE_FILE):
    f.unlink(missing_ok=True)
print(f"saved to {OUT_DIR}:", sorted(p.name for p in OUT_DIR.iterdir()))
print(f"next: 02 full with E5_MODELS=<attached dataset>/{OUT_DIR.name}/{MODEL_DIR.name}, "
      f"then 03 full with BUCKET_DIR=<that run>/embeddings_full/{MODEL_DIR.name}/buckets "
      f"(TRAIN_S1_PER_COUNTRY={LGBM_TRAIN_S1_PER_COUNTRY}, SEED={LGBM_SEED})")